<a href="https://colab.research.google.com/github/vusu4242/hm2_vus_konovalenko/blob/main/notebooks/02_one_stage_retinanet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # One-stage Anchor-based: Retinanet

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

In [3]:
import os
from pathlib import Path
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('hm2')

if not Path('/content/data/train').exists():
    !kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 -p /content/data
    !cd /content/data && unzip -q '*.zip' && rm *.zip
else:
    print('Data is here')
ROOT = Path('/content/data')

100% 9.88G/9.88G [02:09<00:00, 82.1MB/s]



In [4]:
from google.colab import drive
drive.mount('/content/drive')

PREP = Path('/content/prepared')
if not PREP.exists():
    !unzip -q /content/drive/MyDrive/hw2_prepared.zip -d /content
for split in ['train', 'val']:
    n = sum(1 for _ in (PREP / split / 'images').rglob('*.jpg'))
    print(f'{split}: {n} кадрів')

Mounted at /content/drive
train: 4868 кадрів
val: 571 кадрів


## Dataset

Reads the preprocessed frames and YOLO labels , class ids shifted by +1 (0 is reserved for background). Training augmentation: horizontal flip (p=0.5). Normalization (ImageNet mean/std) and resizing are done inside the torchvision model.

In [5]:
import random, cv2, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
SEED = 42

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything()

class DetDataset(Dataset):
    def __init__(self, split, train=False):
        self.imgs = sorted((PREP / split / 'images').rglob('*.jpg'))
        self.train = train

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        p = self.imgs[i]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        lp = Path(str(p).replace('/images/', '/labels/')).with_suffix('.txt')
        a = np.loadtxt(lp, ndmin=2) if lp.stat().st_size else np.zeros((0, 5))
        cls, cx, cy, w, h = a.T
        boxes = np.stack([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H], 1)
        keep = (boxes[:, 2] - boxes[:, 0] > 1) & (boxes[:, 3] - boxes[:, 1] > 1)   # прибираємо вироджені бокси
        boxes, cls = boxes[keep], cls[keep]
        if self.train and random.random() < 0.5:                                    # горизонтальний фліп
            img = img[:, ::-1].copy()
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        target = {'boxes': torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                  'labels': torch.as_tensor(cls + 1, dtype=torch.int64)}            # +1: 0 = фон
        return TF.to_tensor(img), target

def collate(batch):
    return tuple(zip(*batch))

g = torch.Generator().manual_seed(SEED)
train_ds = DetDataset('train', train=True)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate, generator=g)
print('train frames:', len(train_ds))

train frames: 4868


## Evaluation
Predictions are made on the original val frames (every 5th frame, 571 frames, not masked). Metrics are computed with pycocotools (`maxDets=500`); detections inside ignored regions are not penalised thanks to `iscrowd=1` in the ground truth. The same evaluation is used for every model.

In [6]:
import json, contextlib, io, pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

with contextlib.redirect_stdout(io.StringIO()):
    gt = COCO(str(ROOT / 'val/annotations_coco.json'))
imgs_va = pd.DataFrame(gt.dataset['images']).rename(columns={'id': 'image_id'})
val_sub = imgs_va.sort_values(['video', 'frame_id'])
val_sub = val_sub[val_sub.groupby('video').cumcount() % 5 == 0]       # той самий кожен 5-й кадр
print('val frames for evaluation:', len(val_sub))

@torch.no_grad()
def predict(model, df, split='val', bs=4):
    """Передбачення на оригінальних кадрах -> список у COCO-форматі (bbox = [x, y, w, h] у пікселях)."""
    model.eval(); out = []
    for i in range(0, len(df), bs):
        rows = df.iloc[i:i + bs]
        ims = [TF.to_tensor(cv2.cvtColor(cv2.imread(str(ROOT / split / f)), cv2.COLOR_BGR2RGB)).to(DEVICE)
               for f in rows.file_name]
        with torch.autocast('cuda', dtype=torch.float16):
            res = model(ims)
        for iid, r in zip(rows.image_id, res):
            b = r['boxes'].float().cpu().numpy(); b[:, 2:] -= b[:, :2]      # xyxy -> xywh
            for box, s, l in zip(b, r['scores'].float().cpu().numpy(), r['labels'].cpu().numpy()):
                out.append({'image_id': int(iid), 'category_id': int(l),   # мітки 1..8 = COCO category_id
                            'bbox': [round(float(v), 2) for v in box], 'score': float(s)})
    return out

def evaluate(preds, img_ids):
    """COCO mAP + AP по класах."""
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(preds)
        E = COCOeval(gt, dt, 'bbox')
        E.params.imgIds = list(img_ids); E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    res = {'mAP': s[0], 'AP50': s[1], 'AP75': s[2], 'AP_small': s[3], 'AP_medium': s[4], 'AP_large': s[5]}
    prec = E.eval['precision']                                           # [IoU, recall, class, area, maxDets]
    for k, c in enumerate(CLASSES):
        p = prec[:, :, k, 0, -1]
        res[f'AP_{c}'] = float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    return res

val frames for evaluation: 571


### Inference speed
Measures inference speed: batch 1, 1280×720 input, mixed precision; model forward only, image decoding excluded

In [7]:
@torch.no_grad()
def measure_speed(model, df, n=50, warmup=5):
    """Середній час інференсу на кадр (лише forward моделі, без читання картинки)."""
    model.eval()
    ims = [TF.to_tensor(cv2.cvtColor(cv2.imread(str(ROOT / 'val' / f)), cv2.COLOR_BGR2RGB)).to(DEVICE)
           for f in df.file_name.iloc[:n]]
    times = []
    for i, im in enumerate(ims):
        torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast('cuda', dtype=torch.float16):
            model([im])
        torch.cuda.synchronize()
        if i >= warmup:
            times.append(time.time() - t0)
    ms = np.mean(times) * 1000
    return {'ms_per_img': round(ms, 1), 'FPS': round(1000 / ms, 1)}


## Training
Training protocol (the same for the baseline and every experiment):
- SGD, lr 0.005 (linear scaling rule: 0.02 for batch 16 -> 0.005 for batch 4), momentum 0.9, weight decay 1e-4
- linear warm-up for the first 300 iterations, lr × 0.1 after epoch 5
- 6 epochs, batch size 4, mixed precision (AMP), seed 42
- val mAP after every epoch; checkpoint and history are saved to Google Drive after every epoch, so a run can be resumed if Colab disconnects.

In [8]:
CKPT = Path('/content/drive/MyDrive/hw2_runs')
CKPT.mkdir(exist_ok=True)
TRAIN_CFG = dict(epochs=6, batch_size=4, lr=0.005, momentum=0.9, weight_decay=1e-4,
                 warmup_iters=300, lr_drop_epoch=5, amp=True, seed=SEED)


def save_ckpt(path, model, opt, sched, scaler, ep, hist):
    state = dict(model=model.state_dict(), opt=opt.state_dict(),
                 sched=sched.state_dict(), scaler=scaler.state_dict(),
                 epoch=ep, hist=hist)
    torch.save(state, path)


def train_run(name, build_fn=None, patch=None, smoke=False, **model_kw):
    """Train with the shared protocol TRAIN_CFG.
    patch(model, on): loss modification; smoke=True: only 20 iterations."""
    build_fn = build_fn or build_model
    seed_everything()
    g.manual_seed(SEED)
    model = build_fn(**model_kw)
    if patch:
        patch(model, True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=TRAIN_CFG['lr'], momentum=TRAIN_CFG['momentum'],
                          weight_decay=TRAIN_CFG['weight_decay'])
    n_it = len(train_dl)

    def lr_fn(it):
        warm = min(1, (it + 1) / TRAIN_CFG['warmup_iters'])
        drop = 0.1 if it >= TRAIN_CFG['lr_drop_epoch'] * n_it else 1
        return warm * drop

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    scaler = torch.amp.GradScaler('cuda')
    hist, start = [], 0
    ck = CKPT / f'{name}.pt'
    if ck.exists() and not smoke:
        s = torch.load(ck, weights_only=False)
        model.load_state_dict(s['model'])
        opt.load_state_dict(s['opt'])
        sched.load_state_dict(s['sched'])
        scaler.load_state_dict(s['scaler'])
        hist, start = s['hist'], s['epoch'] + 1
        print(f'resume {name} from epoch {start}')
    try:
        for ep in range(start, TRAIN_CFG['epochs']):
            model.train()
            t0, run = time.time(), {}
            for i, (imgs, tg) in enumerate(train_dl):
                imgs = [x.to(DEVICE) for x in imgs]
                tg = [{k: v.to(DEVICE) for k, v in t.items()} for t in tg]
                with torch.autocast('cuda', dtype=torch.float16):
                    losses = model(imgs, tg)
                loss = sum(losses.values())
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                for k, v in losses.items():
                    run[k] = run.get(k, 0) + v.item()
                if smoke and i % 5 == 0:
                    print('  smoke iter', i, {k: round(v.item(), 3) for k, v in losses.items()})
                if smoke and i == 20:
                    print('smoke test OK')
                    return model, None
                if i % 300 == 0:
                    print(f'  epoch {ep} iter {i}/{n_it} loss {loss.item():.3f}')
            train_min = (time.time() - t0) / 60
            m = evaluate(predict(model, val_sub), val_sub.image_id)
            rec = {'epoch': ep, 'train_min': train_min}
            rec.update({k: v / n_it for k, v in run.items()})
            rec.update(m)
            hist.append(rec)
            print(f"epoch {ep}: mAP {m['mAP']:.4f} | AP50 {m['AP50']:.4f} | "
                  f"AP_small {m['AP_small']:.4f} | {train_min:.0f} min")
            save_ckpt(ck, model, opt, sched, scaler, ep, hist)
            json.dump(hist, open(CKPT / f'{name}_hist.json', 'w'), indent=1)
    finally:
        if patch:
            patch(model, False)
    return model, pd.DataFrame(hist)

print('train_run ready')

train_run ready


## Model and training setup


  RetinaNet with a ResNet-50-FPN backbone (torchvision `retinanet_resnet50_fpn_v2`, COCO-pretrained); the classification head is replaced for 8 classes. In this torchvision version the box regression head already uses GIoU loss. Same settings and training protocol as Faster R-CNN: input 1280×720, up to 300 detections per image, score threshold 0.001. Anchors are set through `AnchorGenerator`, keeping 9 anchors per location, so the pretrained regression head is reused. A 20-iteration smoke test checks that the model trains.

In [9]:
import time

In [10]:
from functools import partial
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.anchor_utils import AnchorGenerator
import torchvision.models.detection.retinanet as rn
from torchvision.ops import sigmoid_focal_loss

DEVICE = 'cuda'
BASE_RN = dict(min_size=720, max_size=1280, detections_per_img=300, score_thresh=0.001, topk_candidates=1000)


def make_anchors(base_sizes, ratios):
    sizes = tuple((s, int(s * 2 ** (1 / 3)), int(s * 2 ** (2 / 3))) for s in base_sizes)
    return AnchorGenerator(sizes, (tuple(ratios),) * len(base_sizes))


def build_retinanet(anchor_sizes=(32, 64, 128, 256, 512), ratios=(0.5, 1.0, 2.0), **overrides):
    m = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT, **{**BASE_RN, **overrides})
    m.anchor_generator = make_anchors(anchor_sizes, ratios)
    na = m.anchor_generator.num_anchors_per_location()[0]
    assert na == m.head.classification_head.num_anchors
    m.head.classification_head = RetinaNetClassificationHead(
        256, na, len(CLASSES) + 1, norm_layer=partial(torch.nn.GroupNorm, 32))
    return m.to(DEVICE)


def focal_patch(alpha, gamma):
    def patch(model, on):
        if on:
            rn.sigmoid_focal_loss = partial(sigmoid_focal_loss, alpha=alpha, gamma=gamma)
        else:
            rn.sigmoid_focal_loss = sigmoid_focal_loss
    return patch

_ = train_run('smoke_retina', build_fn=build_retinanet, smoke=True)

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth


100%|██████████| 146M/146M [00:00<00:00, 166MB/s]


  smoke iter 0 {'classification': 1.13, 'bbox_regression': 0.249}
  epoch 0 iter 0/1217 loss 1.379
  smoke iter 5 {'classification': 1.132, 'bbox_regression': 0.294}
  smoke iter 10 {'classification': 1.072, 'bbox_regression': 0.278}
  smoke iter 15 {'classification': 1.034, 'bbox_regression': 0.311}
  smoke iter 20 {'classification': 0.914, 'bbox_regression': 0.243}
smoke test OK


## Baseline

In [ ]:
model_rn_base, hist_rn_base = train_run('retina_baseline', build_fn=build_retinanet)
hist_rn_base[['epoch', 'mAP', 'AP50', 'AP_small']].round(4)

resume retina_baseline from epoch 5
  epoch 5 iter 0/1217 loss 0.311
  epoch 5 iter 300/1217 loss 0.346
  epoch 5 iter 600/1217 loss 0.262
  epoch 5 iter 900/1217 loss 0.311
  epoch 5 iter 1200/1217 loss 0.325
epoch 5: mAP 0.1878 | AP50 0.3608 | AP_small 0.0684 | 11 min


,epoch,mAP,AP50,AP_small
0,0,0.1631,0.3113,0.0632
1,1,0.1738,0.3314,0.0612
2,2,0.1836,0.3515,0.0680
3,3,0.1802,0.3440,0.0689
4,4,0.1818,0.3519,0.0714
5,5,0.1878,0.3608,0.0684


Comparing to Faster R-CNN retina is weaker because:
1. there is no proposal part, which filters out the background-> Retina has to simultaneously classify a lot of anchors and majority of them is background. Focal loss cant compensate it completely because model can confuse background with small objects.
2. The is no the most detailed level of FPN. RetinaNet uses levels P3–P7 (the smallst step  8 px),  Faster R-CNN P2–P6 (step 4 px). For 10–20-px objects this is very essential.

3. Anchors are from 32 px -> objects match poorly with any anchor -> few positive examples during learning.


### Anchor configuration from data
By default, RetinaNet's smallest anchor is 32 px, but at 1280 px more than half of our objects are smaller than that. To check whether the default anchors fit our objects, for every training box we find the anchor shape that overlaps it best (IoU). If even the best anchor has IoU below 0.4, the object has no suitable anchor and is treated as background during training. We compare several anchor sets and use the best one in Experiment 1.

In [ ]:
coco_tr = json.load(open(ROOT / 'train/annotations_coco.json'))
im = pd.DataFrame(coco_tr['images']).set_index('id')
an = pd.DataFrame(coco_tr['annotations']); an = an[an.iscrowd == 0]
scale = 1280 / im.loc[an.image_id, ['width', 'height']].max(axis=1).values
wh = np.array(an.bbox.tolist())[:, 2:] * scale[:, None]                      # розміри боксів при 1280
wh = wh[np.random.RandomState(SEED).choice(len(wh), 200_000, replace=False)]
size = np.sqrt(wh.prod(1))


def anchor_shapes(base_sizes, ratios):
    """Усі форми якорів (w, h), як їх будує torchvision AnchorGenerator (ratio = h/w)."""
    return np.array([(s / np.sqrt(r), s * np.sqrt(r)) for b in base_sizes
                     for s in (b, b * 2 ** (1 / 3), b * 2 ** (2 / 3)) for r in ratios])


def best_iou(wh, anchors):
    iw = np.minimum(wh[:, None, 0], anchors[None, :, 0])
    ih = np.minimum(wh[:, None, 1], anchors[None, :, 1])
    inter = iw * ih
    return (inter / (wh.prod(1)[:, None] + anchors.prod(1)[None] - inter)).max(1)


CANDIDATES = {
    'default 32-512, (0.5,1,2)': ((32, 64, 128, 256, 512), (0.5, 1.0, 2.0)),
    '16-256, (0.5,1,2)':         ((16, 32, 64, 128, 256), (0.5, 1.0, 2.0)),
    '12-192, (0.5,1,2)':         ((12, 24, 48, 96, 192), (0.5, 1.0, 2.0)),
    '16-256, (0.75,1,2)':        ((16, 32, 64, 128, 256), (0.75, 1.0, 2.0)),
}
rows = []
for name, (bs, rt) in CANDIDATES.items():
    iou = best_iou(wh, anchor_shapes(bs, rt))
    rows.append({'anchors': name, 'mean best IoU': iou.mean(),
                 'best IoU < 0.4, %': (iou < .4).mean() * 100,
                 'small (<32px): best IoU < 0.4, %': (iou[size < 32] < .4).mean() * 100,
                 'best IoU >= 0.5, %': (iou >= .5).mean() * 100})
pd.DataFrame(rows).round(3)

,anchors,mean best IoU,"best IoU < 0.4, %","small (<32px): best IoU < 0.4, %","best IoU >= 0.5, %"
0,"default 32-512, (0.5,1,2)",0.648,20.961,38.701,71.904
1,"16-256, (0.5,1,2)",0.811,1.548,2.844,97.682
2,"12-192, (0.5,1,2)",0.822,0.524,0.953,99.126
3,"16-256, (0.75,1,2)",0.787,1.647,2.912,96.722


Default anchors are not suitable for us, as 21% of all objects and 39% of small have no anchor IoU>=0.4 (during learning they are basically background) and only 72% have positive anchors.


Smaller size of anchor is better as for 12-192 99% of objects have positive anchors and without anchor IoU >=0.4 only 0.5% and small - 0.95%-> choose 12-192

Proportions (0.5,1,2) are better -> choose them

## Experiment 1: smaller anchors

In [ ]:
model_rn_exp1, hist_rn_exp1 = train_run('retina_exp1_anchors', build_fn=build_retinanet,
                                        anchor_sizes=(12, 24, 48, 96, 192))
hist_rn_exp1[['epoch', 'mAP', 'AP50', 'AP_small']].round(4)

  epoch 0 iter 0/1217 loss 1.734
  epoch 0 iter 300/1217 loss 0.788
  epoch 0 iter 600/1217 loss 0.550
  epoch 0 iter 900/1217 loss 0.567
  epoch 0 iter 1200/1217 loss 0.572
epoch 0: mAP 0.1136 | AP50 0.2724 | AP_small 0.0536 | 11 min
  epoch 1 iter 0/1217 loss 0.579
  epoch 1 iter 300/1217 loss 0.640
  epoch 1 iter 600/1217 loss 0.562
  epoch 1 iter 900/1217 loss 0.591
  epoch 1 iter 1200/1217 loss 0.508
epoch 1: mAP 0.1358 | AP50 0.3140 | AP_small 0.0580 | 11 min
  epoch 2 iter 0/1217 loss 0.548
  epoch 2 iter 300/1217 loss 0.589
  epoch 2 iter 600/1217 loss 0.475
  epoch 2 iter 900/1217 loss 0.507
  epoch 2 iter 1200/1217 loss 0.445
epoch 2: mAP 0.1377 | AP50 0.3249 | AP_small 0.0605 | 11 min
  epoch 3 iter 0/1217 loss 0.555
  epoch 3 iter 300/1217 loss 0.495
  epoch 3 iter 600/1217 loss 0.462
  epoch 3 iter 900/1217 loss 0.445
  epoch 3 iter 1200/1217 loss 0.516
epoch 3: mAP 0.1448 | AP50 0.3372 | AP_small 0.0620 | 11 min
  epoch 4 iter 0/1217 loss 0.412
  epoch 4 iter 300/1217 los

,epoch,mAP,AP50,AP_small
0,0,0.1136,0.2724,0.0536
1,1,0.1358,0.3140,0.0580
2,2,0.1377,0.3249,0.0605
3,3,0.1448,0.3372,0.0620
4,4,0.1470,0.3397,0.0605
5,5,0.1511,0.3462,0.0601


In [ ]:
for name in ['retina_baseline', 'retina_exp1_anchors']:
    h = json.load(open(CKPT / f'{name}_hist.json'))[-1]
    print(name, {k: round(h[k], 4) for k in ['mAP', 'AP_small', 'AP_medium', 'AP_large']})

retina_baseline {'mAP': 0.1878, 'AP_small': 0.0684, 'AP_medium': 0.1654, 'AP_large': 0.3683}
retina_exp1_anchors {'mAP': 0.1511, 'AP_small': 0.0601, 'AP_medium': 0.1397, 'AP_large': 0.2837}


The results are worse - mAP 0.150 vs 0.188, AP_small 0.060 vs 0.068.

Possibly:
1. Anchor coverage was not the main problem for small objects. The finest RetinaNet level (P3, stride 8 px) is too coarse: a 10–15 px object covers only 1–2 feature cells, so the model sees it poorly with any anchor. Faster R-CNN uses P2 (stride 4 px) and reaches AP_small 0.116.
2. The pretrained model was trained with the default anchors, so each FPN level is used to certain object sizes. Shrinking all anchors ~2.7× changes this, and 6 epochs are not enough to adapt: mAP starts much lower (0.114 vs 0.163 after epoch 0) and is still growing at the end.
3. The largest anchor became ~305 px, so large objects match anchors worse, which explains the biggest drop in AP_large.


So experiment2 on standart sizes

## Experiment 2: focal loss γ=1.5 (default anchors)

Experiment 2 is built on the default anchors. Focal loss down-weights easy examples; γ controls how strongly. Our frames have many easy background anchors but also a strong class imbalance, so γ is lowered from 2 to 1.5 (less down-weighting) to check whether the balance of easy and hard examples limits the model.

In [ ]:
model_rn_exp2, hist_rn_exp2 = train_run('retina_exp2_focal', build_fn=build_retinanet,
                                        patch=focal_patch(0.25, 1.5))
hist_rn_exp2[['epoch', 'mAP', 'AP50', 'AP_small']].round(4)

  epoch 0 iter 0/1217 loss 1.435
  epoch 0 iter 300/1217 loss 0.874
  epoch 0 iter 600/1217 loss 0.494
  epoch 0 iter 900/1217 loss 0.513
  epoch 0 iter 1200/1217 loss 0.594
epoch 0: mAP 0.1658 | AP50 0.3143 | AP_small 0.0632 | 11 min
  epoch 1 iter 0/1217 loss 0.514
  epoch 1 iter 300/1217 loss 0.556
  epoch 1 iter 600/1217 loss 0.521
  epoch 1 iter 900/1217 loss 0.493
  epoch 1 iter 1200/1217 loss 0.501
epoch 1: mAP 0.1748 | AP50 0.3327 | AP_small 0.0636 | 11 min
  epoch 2 iter 0/1217 loss 0.507
  epoch 2 iter 300/1217 loss 0.547
  epoch 2 iter 600/1217 loss 0.423
  epoch 2 iter 900/1217 loss 0.485
  epoch 2 iter 1200/1217 loss 0.486
epoch 2: mAP 0.1803 | AP50 0.3478 | AP_small 0.0673 | 11 min
  epoch 3 iter 0/1217 loss 0.457
  epoch 3 iter 300/1217 loss 0.451
  epoch 3 iter 600/1217 loss 0.469
  epoch 3 iter 900/1217 loss 0.386
  epoch 3 iter 1200/1217 loss 0.434
epoch 3: mAP 0.1773 | AP50 0.3429 | AP_small 0.0777 | 11 min
  epoch 4 iter 0/1217 loss 0.371
  epoch 4 iter 300/1217 los

,epoch,mAP,AP50,AP_small
0,0,0.1658,0.3143,0.0632
1,1,0.1748,0.3327,0.0636
2,2,0.1803,0.3478,0.0673
3,3,0.1773,0.3429,0.0777
4,4,0.1863,0.3615,0.0737
5,5,0.1850,0.3542,0.0682


The result is the same as the baseline (mAP 0.185 vs 0.188, AP_small 0.068 vs 0.068); the difference is within the epoch-to-epoch noise.

Together with Experiment 1, this suggests that RetinaNet is limited by too coarse features for small objects (finest level P3, stride 8 px) rather than by its anchors or loss.

### Comparison with Faster R-CNN
Final val metrics of all RetinaNet runs and the best Faster R-CNN (same data, protocol and evaluation).

In [11]:
import json
rows = []
for name in ['frcnn_baseline', 'retina_baseline', 'retina_exp1_anchors', 'retina_exp2_focal']:
    h = json.load(open(CKPT / f'{name}_hist.json'))[-1]
    rows.append({'run': name, **{k: round(h[k], 4) for k in
                 ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']}})
table = pd.DataFrame(rows)
display(table)

h_rn = json.load(open(CKPT / 'retina_baseline_hist.json'))[-1]
h_fr = json.load(open(CKPT / 'frcnn_baseline_hist.json'))[-1]
display(pd.DataFrame({'retina_baseline': {c: round(h_rn[f'AP_{c}'], 4) for c in CLASSES},
                      'frcnn_baseline': {c: round(h_fr[f'AP_{c}'], 4) for c in CLASSES}}))
table.to_csv(CKPT / 'retina_table.csv', index=False)

model_rn_base, _ = train_run('retina_baseline', build_fn=build_retinanet)
print('RetinaNet speed:', measure_speed(model_rn_base, val_sub))

,run,mAP,AP50,AP75,AP_small,AP_medium,AP_large
0,frcnn_baseline,0.2303,0.4530,0.2073,0.1162,0.2161,0.4014
1,retina_baseline,0.1878,0.3608,0.1724,0.0684,0.1654,0.3683
2,retina_exp1_anchors,0.1511,0.3462,0.1120,0.0601,0.1397,0.2837
3,retina_exp2_focal,0.1850,0.3542,0.1735,0.0682,0.1651,0.3744


,retina_baseline,frcnn_baseline
person,0.2413,0.2639
bicycle,0.1498,0.2154
car,0.3752,0.3821
van,0.2193,0.2509
truck,0.2092,0.3022
tricycle,0.1657,0.2356
bus,0.0557,0.0783
motor,0.0863,0.1140


resume retina_baseline from epoch 6
RetinaNet speed: {'ms_per_img': np.float64(48.3), 'FPS': np.float64(20.7)}


1. Faster R-CNN is better on all metrics (mAP 0.230 vs 0.188; tuned Faster R-CNN with 2000 test proposals: 0.232), especially on small objects (AP_small 0.116 vs 0.068). RetinaNet has no proposal stage, so it does not filter out background before classification and refines each box only once; in addition, it lacks the finest P2 feature level.
2. The gap is largest for rare and confusable classes (truck 0.209 vs 0.302, tricycle 0.166 vs 0.236, bicycle 0.150 vs 0.215) and almost zero for the frequent, larger car class (0.375 vs 0.382).
3. Speed: RetinaNet is about 1.6× faster (48 ms / 21 FPS vs ~74–80 ms / ~13 FPS for Faster R-CNN), because it has no second stage that processes each proposal separately. So RetinaNet trades accuracy for speed.
4. These issues can be addressed with focal loss (already used in RetinaNet), better positive sample assignment such as ATSS, a finer P2 level or higher input resolution for small objects, or anchor-free detectors like FCOS and YOLO.